# Notebook 06: offline password guessing against PARAM II

**Finding.** Fig. 3's attack 1 (offline password guessing) is supposed to fail against a semi-honest server that sees one blinded TOPRF request $b = a r + a_x + 2e$ (Fig. 2). It does not at PARAM II as this code runs it.

Table II gives PARAM II the degree $n = 595$. This implementation runs it in the ring of the scheme text, $R_q = \mathbb{Z}_q[X]/(X^{n}+1)$, so $\mathbb{Z}_q[X]/(X^{595}+1)$ (plan decision 1). The paper's security estimate for $n = 595$ likely assumes plain LWE, where this attack does not apply, so the flaw belongs to that reading of the parameters and is not proven of the paper. Since 595 is odd, $(-1)^{595} + 1 = 0$, so $X = -1$ is a root and

$$\varphi(a) = a(-1) = \sum_i (-1)^i a_i \bmod q$$

is a ring homomorphism $R_q \to \mathbb{Z}_q$. Applying it to each of the $\ell$ entries of $b$ and subtracting $\varphi(a_{x'})$ for a guess $x'$ gives

$$c_j = \varphi(b_j) - \varphi(a_{x',j}) = \varphi(a_j)\,\varphi(r) + 2\varphi(e_j) + \varphi(a_{x,j} - a_{x',j}).$$

For the real password the last term is 0, and $\varphi(r)$, $\varphi(e_j)$ are sums of 595 Gaussians, about $\sigma\sqrt{n} \approx 78$ in size. The server tries every $\rho \in [-R, R]$ and keeps the guess if $c_j - \rho\,\varphi(a_j)$ is small in every row $j$. For a wrong $x'$ the last term is uniform in $\mathbb{Z}_q$, so no $\rho$ fits.

PARAM I uses $X^{512}+1$, which has no root mod $q$ ($1024 \nmid q - 1$), so $\varphi$ is not a homomorphism and the same test finds nothing. The attack code is `ris_sem_project.root_attack`. This notebook only calls it.

In [1]:
import time

import numpy as np

from ris_sem_project import protocol
from ris_sem_project.params import PARAM_I, PARAM_II, Lattice, setup
from ris_sem_project.parties.qpase_user import QPASEUser
from ris_sem_project.root_attack import DEFAULT_RADIUS, evaluate_at_minus_one, match_threshold, offline_password_test

PASSWORD = "sunshine1"
DICTIONARY = [
    "123456", "password", "12345678", "qwerty", "123456789", "12345", "1234", "111111",
    "1234567", "dragon", "123123", "baseball", "abc123", "football", "monkey", "letmein",
    "shadow", "master", "666666", "qwertyuiop", "123321", "mustang", "1234567890", "michael",
    "sunshine1", "iloveyou", "trustno1", "princess",
]
assert PASSWORD in DICTIONARY


def deploy(params):
    """N = 3 servers with threshold t = 2, and a user who registers with PASSWORD."""
    pp = setup(params, N=3, t=2, mu=3)
    servers = protocol.setup_servers(pp)
    user = QPASEUser(pp, "alice", PASSWORD)
    protocol.register(user, servers)
    return pp, servers, user


pp2, servers2, user2 = deploy(PARAM_II)
ring2 = Lattice.from_pp(pp2).ring
print(f"PARAM II: q = {pp2.q}, n = {pp2.n} (odd: {pp2.n % 2 == 1}), ell = {pp2.ell}, sigma = {pp2.sigma}")
print(f"(-1)^n + 1 mod q = {((-1) ** pp2.n + 1) % pp2.q}, so X = -1 is a root of X^n + 1")
print(f"alice registered at {len(servers2)} servers")

PARAM II: q = 268435399, n = 595 (odd: True), ell = 28, sigma = 3.2
(-1)^n + 1 mod q = 0, so X = -1 is a root of X^n + 1
alice registered at 3 servers


In [2]:
requests = user2.toprf_request([1, 2])
request = requests[1]
print(f"S_1 receives user_id = {request.user_id!r} and {len(request.blinded_input)} bytes of b")

true_phi_r = int(ring2.centered(evaluate_at_minus_one(ring2, user2._phase.r)))
tau = match_threshold(pp2)
print(f"sigma sqrt(n) = {pp2.sigma * np.sqrt(pp2.n):.1f}: the spread of phi(r) and of phi(e_j)")
print(f"search radius R = {DEFAULT_RADIUS}, match threshold tau = {tau} (= 2 * 8 * sigma sqrt(n))")
print(f"a random residual is near q/4 = {pp2.q // 4}, which is {pp2.q // 4 // tau}x tau")

S_1 receives user_id = 'alice' and 66640 bytes of b
sigma sqrt(n) = 78.1: the spread of phi(r) and of phi(e_j)
search radius R = 1500, match threshold tau = 1249 (= 2 * 8 * sigma sqrt(n))
a random residual is near q/4 = 67108849, which is 53730x tau


In [3]:
start = time.perf_counter()
results = offline_password_test(pp2, request, DICTIONARY)
elapsed = time.perf_counter() - start

print(f"{'guess':<14}{'residual':>12}  {'matched':<8}recovered phi(r)")
for r in sorted(results, key=lambda r: r.residual):
    print(f"{r.password:<14}{r.residual:>12}  {str(r.matched):<8}{r.phi_r}")
print(f"{len(DICTIONARY)} guesses against one request in {elapsed:.1f} s")

guess             residual  matched recovered phi(r)
sunshine1              326  True    44
mustang           89915733  False   None
master            95009118  False   None
123456789         95946994  False   None
trustno1          96125880  False   None
princess          96218008  False   None
baseball          97968956  False   None
1234567           97987418  False   None
123123            98103930  False   None
12345678          99154772  False   None
letmein           99296298  False   None
iloveyou          99633157  False   None
123321            99698583  False   None
dragon            99912791  False   None
1234             100638404  False   None
111111           101151755  False   None
monkey           101730792  False   None
football         101753953  False   None
shadow           102303453  False   None
abc123           103250558  False   None
123456           104075777  False   None
michael          104195883  False   None
666666           104668834  False   None
qwerty

In [4]:
hit = next(r for r in results if r.matched)
wrong = [r for r in results if r.password != PASSWORD]
print(f"matched guesses: {[r.password for r in results if r.matched]}")
print(f"true password residual:        {hit.residual}")
print(f"smallest wrong-guess residual: {min(r.residual for r in wrong)}")
print(f"largest wrong-guess residual:  {max(r.residual for r in wrong)}")
print(f"recovered phi(r) = {hit.phi_r}, user's phi(r) = {true_phi_r}")

matched guesses: ['sunshine1']
true password residual:        326
smallest wrong-guess residual: 89915733
largest wrong-guess residual:  108321016
recovered phi(r) = 44, user's phi(r) = 44


In [5]:
pp1, servers1, user1 = deploy(PARAM_I)
request1 = user1.toprf_request([1, 2])[1]
tau1 = match_threshold(pp1)

control = offline_password_test(pp1, request1, DICTIONARY)
print(f"PARAM I: n = {pp1.n}, match threshold tau = {tau1}")
print(f"matched guesses: {[r.password for r in control if r.matched]}")
print(f"residual of the real password:  {next(r.residual for r in control if r.password == PASSWORD)}")
print(f"smallest residual of any guess: {min(r.residual for r in control)}")

PARAM I: n = 512, match threshold tau = 1159
matched guesses: []
residual of the real password:  94906142
smallest residual of any guess: 86117436


In [6]:
assert len(DICTIONARY) >= 20
assert [r.password for r in results if r.matched] == [PASSWORD]
assert hit.phi_r == true_phi_r
assert hit.residual < tau
assert all(not r.matched and r.phi_r is None for r in wrong)
assert all(r.residual > 100 * tau for r in wrong)
assert not any(r.matched for r in control)
assert all(r.residual > 100 * tau1 for r in control)
print("PARAM II: the real password is the only match and phi(r) is recovered exactly.")
print("PARAM I: no guess matches, not even the real password.")

PARAM II: the real password is the only match and phi(r) is recovered exactly.
PARAM I: no guess matches, not even the real password.
